# Build a CECL lifetime loss model for residential mortgages

This notebook builds a *Current Expected Credit Loss* (CECL) model for a portfolio of first-lien residential mortgages and runs it at a quarter-end reporting date, ready to be documented and validated with the ValidMind Library.

Under CECL (FASB ASC 326), a lender reserves on day one for the credit losses it expects over the **full contractual life** of each loan, net of expected prepayments, using information about past events, current conditions and **reasonable and supportable forecasts** of the economy. Beyond the forecast horizon the estimate **reverts** to historical loss experience.

The model follows the approach most US banks use for mortgage portfolios — a loan-level, macro-conditioned *PD × LGD × EAD* projection:

| Component | Method |
|---|---|
| **Default and prepayment** | Quarterly discrete-time competing-risk hazards, one gradient-boosted classifier per cause (cause-specific hazards). Each quarter a loan defaults, prepays or stays current, driven by credit score, mark-to-market LTV, DTI, occupancy, purpose, term, loan age, the unemployment rate and the refinance incentive. Monotonic constraints keep the economically signed drivers pointing the right way. |
| **Loss given default (LGD)** | Gradient-boosted regressor on realized severities, constrained to rise with mark-to-market LTV at default. |
| **Exposure at default (EAD)** | Contractual amortization schedule. Prepayment is handled by the hazard model, not by EAD. |
| **Macroeconomic forecast** | Federal Reserve 2025 supervisory scenarios, re-based on the reporting-quarter actuals: baseline, adverse and severely adverse, probability-weighted. |
| **Challenger** | The same three components as logistic hazards and a linear LGD on piecewise-linear shapes of the drivers — the transparent benchmark the champion is held against, run through the same lifetime engine. |
| **Reasonable and supportable period** | 8 quarters, then a 4-quarter straight-line reversion of the macro inputs to their long-run averages since 1990, held there for the remaining life. |

**Data.** The macroeconomic history is real: FRED unemployment rate (`UNRATE`), S&P CoreLogic Case-Shiller U.S. National Home Price Index (`CSUSHPINSA`) and the Freddie Mac 30-year mortgage rate (`MORTGAGE30US`), 1990–2026. The loan portfolio is synthetic: about 27,000 mortgages originated 2004–2026 and walked quarter by quarter through that real history, so the Great Financial Crisis, the 2012 and 2020–21 refinance waves and the post-2022 rate lock-in all appear in the performance data. Two features of the real experience are built in that the loan tape does not show — unreported low-documentation layering in 2004–2007 vintages, and forbearance that held defaults down in 2020–21 — so the model faces the same blind spots a bank's model does. Performance through 2019 is split by loan into **train** and **test** samples; 2020 onwards is **out of time**.

::: {.content-hidden when-format="html"}
## Contents
- [Setting up](#toc1__)
- [Load the data](#toc2__)
  - [Macroeconomic history](#toc2_1__)
  - [Loan portfolio](#toc2_2__)
  - [Historical performance](#toc2_3__)
- [Split the samples](#toc3__)
- [Load the models](#toc4__)
  - [Default and prepayment](#toc4_1__)
  - [Loss given default](#toc4_2__)
- [Build the macroeconomic scenarios](#toc5__)
- [Estimate the allowance](#toc6__)
  - [Reporting-date portfolio](#toc6_1__)
  - [Lifetime expected credit loss](#toc6_2__)
  - [Allowance by segment](#toc6_3__)
  - [Champion and challenger allowance](#toc6_4__)
- [Initialize the ValidMind inputs](#toc7__)
- [Run the CECL tests](#toc8__)
  - [Default and prepayment models](#toc8_1__)
  - [LGD model](#toc8_2__)
  - [Lifetime loss engine](#toc8_3__)
  - [Allowance assumptions](#toc8_4__)
  - [Stress testing](#toc8_5__)
  - [Challenger model](#toc8_6__)
:::

<a id='toc1__'></a>

## Setting up

### Install the ValidMind Library

In [ ]:
%pip install -q validmind

### Initialize the ValidMind Library

Register a model in the ValidMind Platform inventory, then paste the code snippet from its **Getting Started** page below — or keep your credentials in an `.env` file next to this notebook (`VM_API_HOST`, `VM_API_KEY`, `VM_API_SECRET`, `VM_API_MODEL`), which the cell loads first.

In [ ]:
# Load your model identifier credentials from an `.env` file

%load_ext dotenv
%dotenv .env

# Or replace with your code snippet

import validmind as vm

vm.init(
    # api_host="...",
    # api_key="...",
    # api_secret="...",
    # model="...",
    document="documentation",
)

### Initialize the Python environment

Three modules sit next to this notebook: `cecl_data.py` loads the macroeconomic data and simulates the loan portfolio, `cecl_model.py` holds the model's feature preparation, the scenario builder and the lifetime loss engine, and `cecl_train.py` splits the samples and fits the champion and challenger models. Keeping them in modules lets other notebooks load the same models, and lets the CECL tests re-run exactly the calculation this notebook books.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score

from cecl_data import REPORTING_QUARTER, load_macro_history, simulate_portfolio
from cecl_model import (
    RS_QUARTERS,
    REVERSION_QUARTERS,
    SCENARIO_WEIGHTS,
    build_scenario_paths,
    long_run_averages,
    model_features,
    portfolio_at,
    segments,
    weighted_allowance,
)
from cecl_train import DEFAULT_DIRECTION, DEV_END, PREPAY_DIRECTION, load_models, split_samples

pd.set_option("display.float_format", "{:,.4f}".format)

COLORS = {
    "actual": "#083E44",
    "predicted": "#DE257E",
    "champion": "#DE257E",
    "challenger": "#F2A541",
    "baseline": "#083E44",
    "adverse": "#F2A541",
    "severely_adverse": "#DE257E",
}

<a id='toc2__'></a>

## Load the data

<a id='toc2_1__'></a>

### Macroeconomic history

Quarterly averages of the three drivers the model uses, from 1990Q1 to the reporting quarter, 2026Q3.

In [ ]:
macro = load_macro_history()

MACRO_TITLES = {
    "unemployment_rate": "Unemployment rate (%)",
    "house_price_index": "Case-Shiller national HPI",
    "mortgage_rate": "30-year mortgage rate (%)",
}
fig = make_subplots(rows=1, cols=3, subplot_titles=list(MACRO_TITLES.values()))
for i, col in enumerate(MACRO_TITLES, start=1):
    fig.add_trace(
        go.Scatter(x=macro.index.to_timestamp(), y=macro[col], line_color=COLORS["actual"]),
        row=1,
        col=i,
    )
fig.update_layout(height=320, showlegend=False, margin=dict(t=40, b=20))
fig.show()

macro.tail()

<a id='toc2_2__'></a>

### Loan portfolio

`simulate_portfolio` returns two tables:

- **`loans`** — the loan tape: one row per mortgage with its origination attributes and final status.
- **`panel`** — the performance history: one row per loan per quarter while the loan was on the books. `outcome` is what happened that quarter (`0` current, `1` default, `2` prepay); `balance` is the scheduled balance at the start of the quarter, `current_ltv` the balance over the house-price-indexed property value, and `refi_incentive` the note rate minus the market mortgage rate. Default rows carry the realized `lgd` and `loss`.

In [ ]:
loans, panel = simulate_portfolio(seed=42)

print(f"{len(loans):,} loans, {len(panel):,} loan-quarters")
loans["status"].value_counts()

In [ ]:
loans.head()

In [ ]:
panel.head()

<a id='toc2_3__'></a>

### Historical performance

Annualized default rate, conditional prepayment rate (CPR) and loss rate by calendar year. The crisis peak, the refinance waves of 2012 and 2020–21, and the collapse in prepayments once rates rose in 2022 are the portfolio's response to the real macro history above.

In [ ]:
by_year = panel.groupby(panel["quarter"].dt.year)
history = pd.DataFrame(
    {
        "loan_quarters": by_year.size(),
        "default_rate": by_year["outcome"].apply(lambda s: (s == 1).mean() * 4),
        "cpr": by_year["outcome"].apply(lambda s: 1 - (1 - (s == 2).mean()) ** 4),
        "loss_rate": by_year["loss"].sum() / by_year["balance"].sum() * 4,
        "mean_lgd": by_year["lgd"].mean(),
    }
)

rates = ["default_rate", "cpr", "loss_rate"]
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{c} (% annualized)" for c in rates])
for i, col in enumerate(rates, start=1):
    fig.add_trace(
        go.Bar(x=history.index, y=history[col] * 100, marker_color=COLORS["actual"]),
        row=1,
        col=i,
    )
fig.update_layout(height=320, showlegend=False, margin=dict(t=40, b=20))
fig.show()

history

<a id='toc3__'></a>

## Split the samples

`split_samples` in `cecl_train.py` cuts the loan-quarter panel three ways:

| Sample | Quarters | Used for |
|---|---|---|
| **Train** | 2004Q1–2019Q4, 70% of loans | Fitting the models |
| **Test** | 2004Q1–2019Q4, the other 30% of loans | In-time performance on loans the models have not seen |
| **Out of time** | 2020Q1 onwards | The pandemic, the refinance boom and the rate shock — conditions the models were not fitted on |

The in-time split is **by loan**, not by row: quarters of the same loan are correlated, so a row split would put near-copies of training rows in the test sample. Quarters in which a loan reached contractual maturity are excluded, since that payoff is scheduled, not a prepayment decision.

In [ ]:
samples = split_samples(panel)
train_panel, test_panel, oot_panel = samples["train"], samples["test"], samples["oot"]
train_defaults, test_defaults, oot_defaults = (
    samples["train_defaults"], samples["test_defaults"], samples["oot_defaults"]
)

pd.DataFrame(
    {
        name: {"loan_quarters": len(df), "loans": df["loan_id"].nunique(), "defaults": df["defaulted"].sum(), "prepayments": df["prepaid"].sum()}
        for name, df in [("train", train_panel), ("test", test_panel), ("out_of_time", oot_panel)]
    }
).T

<a id='toc4__'></a>

## Load the models

The models are fitted in `cecl_train.py` on the train sample and saved to `models/cecl_models.joblib`. `load_models` reads that file, and refits and saves the models first if the file is missing, was saved with another scikit-learn version, or is older than the code — a pickled scikit-learn model is only reliable on the version that saved it. Other notebooks load the same models the same way.

There are two model sets, each with a default, a prepayment and an LGD model taking the same nine loan and macro features (`FEATURES` and `model_features` in `cecl_model.py`):

| | Champion | Challenger |
|---|---|---|
| **Default, prepayment** | Gradient-boosted classifiers with monotonic constraints | Logistic regressions on piecewise-linear shapes of the drivers |
| **LGD** | Shallow gradient-boosted regressor, rising with LTV | Ridge regression on the same shapes |
| **Beyond the training range** | Flat: the trees repeat their edge value | Linear: the hinge terms keep their last slope |

<a id='toc4_1__'></a>

### Default and prepayment

Default and prepayment are competing risks: a loan that prepays can no longer default. Each model set estimates them as **cause-specific hazards** — one classifier predicting the probability of default in the quarter and one the probability of prepayment. The lifetime projection combines them through the survival recursion in [Lifetime expected credit loss](#toc6_2__).

The champion's gradient boosting finds the non-linear shapes and interactions a logit needs hand-built (the seasoning hump, the refinance S-curve, negative equity), while **monotonic constraints** keep the drivers with a known economic sign from flipping in sparse regions — a requirement for a model that will be run under stress scenarios:

| Model | Increasing in | Decreasing in |
|---|---|---|
| Default | current LTV, DTI, unemployment rate | FICO |
| Prepayment | refinance incentive | current LTV |

The challenger builds those shapes by hand: degree-1 splines with knots at the quantiles of each numeric driver, which are hinge functions a logistic regression can weigh. It has no monotonic constraints, so its signs are an empirical result for the tests to check.

In [ ]:
champion, challenger = load_models()
model_sets = {"champion": champion, "challenger": challenger}

pd.DataFrame(
    {
        (set_name, event): {
            sample: roc_auc_score(df[target], models[event].predict_proba(model_features(df))[:, 1])
            for sample, df in [("train", train_panel), ("test", test_panel), ("out_of_time", oot_panel)]
        }
        for set_name, models in model_sets.items()
        for event, target in [("default", "defaulted"), ("prepay", "prepaid")]
    }
).rename_axis("AUC")

Predicted against actual quarterly default and prepayment rates, in time (train and test) and out of time. The 2020 gap is forbearance: unemployment spiked but defaults did not follow, and nothing in the loan tape says why. The challenger, extrapolating linearly in unemployment, over-predicts that spike far more than the champion.

In [ ]:
history_panel = pd.concat([train_panel, test_panel, oot_panel])
X_all = model_features(history_panel)
by_quarter = pd.DataFrame(
    {"actual_default": history_panel["defaulted"], "actual_prepay": history_panel["prepaid"]}
    | {
        f"{set_name}_{event}": models[event].predict_proba(X_all)[:, 1]
        for set_name, models in model_sets.items()
        for event in ["default", "prepay"]
    }
).groupby(history_panel["quarter"].to_numpy()).mean() * 400
quarters = pd.PeriodIndex(by_quarter.index).to_timestamp()

fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Default rate (% annualized)", "Prepayment rate (% annualized)"]
)
for i, event in enumerate(["default", "prepay"], start=1):
    for kind in ["actual", "champion", "challenger"]:
        fig.add_trace(
            go.Scatter(
                x=quarters, y=by_quarter[f"{kind}_{event}"], name=kind,
                line_color=COLORS[kind], showlegend=i == 1,
            ),
            row=1,
            col=i,
        )
    fig.add_vline(x=(DEV_END + 1).to_timestamp(), line_dash="dash", line_color="grey", row=1, col=i)
fig.update_layout(height=360, margin=dict(t=40, b=20), legend=dict(orientation="h", y=-0.15))
fig.show()

<a id='toc4_2__'></a>

### Loss given default

Severity is learned from the defaults in the train sample. The champion's gradient-boosted regressor is constrained to rise with mark-to-market LTV at default — a borrower in negative equity leaves the lender a collateral shortfall on top of foreclosure and sale costs — and with only a few hundred defaults its trees are kept shallow. Predictions from both models are clipped to $[0, 1]$.

In [ ]:
def predict_lgd(model, df):
    return model.predict(model_features(df)).clip(0, 1)


pd.DataFrame(
    {
        (set_name, sample): {
            "defaults": len(d),
            "actual_mean_lgd": d["lgd"].mean(),
            "predicted_mean_lgd": predict_lgd(models["lgd"], d).mean(),
            "correlation": np.corrcoef(d["lgd"], predict_lgd(models["lgd"], d))[0, 1],
        }
        for set_name, models in model_sets.items()
        for sample, d in [("train", train_defaults), ("test", test_defaults), ("out_of_time", oot_defaults)]
    }
)

<a id='toc5__'></a>

## Build the macroeconomic scenarios

The Federal Reserve's 2025 supervisory scenarios start from 2024Q4. To use them at a 2026Q3 reporting date, each scenario's **change from its own jump-off quarter** is applied to the 2026Q3 actuals — the shape of the shock is kept, the starting point is current. The Fed no longer publishes an adverse scenario, so *adverse* is the midpoint of baseline and severely adverse.

Each path then runs:

1. **Quarters 1–8** — the reasonable and supportable forecast.
2. **Quarters 9–12** — straight-line reversion to the long-run averages since 1990 (unemployment and mortgage rate levels, house-price growth rate).
3. **Quarter 13 onwards** — long-run averages for the remaining contractual life.

In [ ]:
reporting = pd.Period(REPORTING_QUARTER)
long_run = long_run_averages()
scenario_paths = build_scenario_paths()

fig = make_subplots(rows=1, cols=3, subplot_titles=list(MACRO_TITLES.values()))
recent = macro.loc["2018Q1":]
for i, col in enumerate(MACRO_TITLES, start=1):
    fig.add_trace(
        go.Scatter(
            x=recent.index.to_timestamp(), y=recent[col], name="actual",
            line_color="grey", showlegend=i == 1,
        ),
        row=1,
        col=i,
    )
    for s, path in scenario_paths.items():
        head = path.iloc[:20]
        fig.add_trace(
            go.Scatter(
                x=head.index.to_timestamp(), y=head[col], name=s,
                line_color=COLORS[s], showlegend=i == 1,
            ),
            row=1,
            col=i,
        )
    fig.add_vrect(
        x0=(reporting + 1).to_timestamp(), x1=(reporting + RS_QUARTERS + 1).to_timestamp(),
        fillcolor="grey", opacity=0.08, line_width=0, row=1, col=i,
    )
fig.update_layout(height=340, margin=dict(t=40, b=20), legend=dict(orientation="h", y=-0.15))
fig.show()

pd.DataFrame(long_run, index=["long-run average since 1990"])

<a id='toc6__'></a>

## Estimate the allowance

<a id='toc6_1__'></a>

### Reporting-date portfolio

The loans still on the books at the end of 2026Q3, with their balance rolled forward to the start of the first forecast quarter.

In [ ]:
portfolio = portfolio_at(panel, REPORTING_QUARTER)

print(f"{len(portfolio):,} loans, amortized cost ${portfolio['balance'].sum() / 1e9:,.2f}bn")
portfolio[["fico", "orig_ltv", "current_ltv", "dti", "note_rate", "loan_age", "balance"]].describe()

<a id='toc6_2__'></a>

### Lifetime expected credit loss

For each loan $i$ and forecast quarter $h$ up to contractual maturity, under scenario $s$:

$$\text{ECL}_{i,s} = \sum_{h=1}^{H_i} S_{i,h-1} \cdot PD_{i,h} \cdot LGD_{i,h} \cdot EAD_{i,h}, \qquad S_{i,h} = S_{i,h-1}\,(1 - PD_{i,h} - PP_{i,h})$$

where $PD$ and $PP$ are the quarterly default and prepayment hazards, $S$ is the probability the loan is still on the books, $LGD$ is evaluated at the projected mark-to-market LTV, and $EAD$ is the scheduled balance. The allowance is the probability-weighted ECL across scenarios. Losses are not discounted, as is common for a non-DCF PD/LGD method under ASC 326-20-30-5.

`lifetime_ecl` in `cecl_model.py` implements this for all loans and quarters at once; `weighted_allowance` runs it per scenario and applies the scenario weights.

In [ ]:
MODEL_ORDER = ["default", "prepay", "lgd"]  # the order the lifetime engine takes them in
models = tuple(champion[m] for m in MODEL_ORDER)
allowance, ecl_results = weighted_allowance(portfolio, scenario_paths, models)

for s, result in ecl_results.items():
    portfolio[f"ecl_{s}"] = result["ecl"]
    portfolio[f"lifetime_pd_{s}"] = result["lifetime_pd"]
portfolio["ecl"] = sum(w * portfolio[f"ecl_{s}"] for s, w in SCENARIO_WEIGHTS.items())

In [ ]:
amortized_cost = portfolio["balance"].sum()
summary = pd.DataFrame(
    {
        s: {
            "weight": SCENARIO_WEIGHTS[s],
            "lifetime_pd": np.average(portfolio[f"lifetime_pd_{s}"], weights=portfolio["balance"]),
            "ecl": portfolio[f"ecl_{s}"].sum(),
            "allowance_ratio": portfolio[f"ecl_{s}"].sum() / amortized_cost,
        }
        for s in SCENARIO_WEIGHTS
    }
).T
summary.loc["probability_weighted"] = [
    1.0,
    np.nan,
    portfolio["ecl"].sum(),
    portfolio["ecl"].sum() / amortized_cost,
]
summary.style.format(
    {"weight": "{:.0%}", "lifetime_pd": "{:.2%}", "ecl": "${:,.0f}", "allowance_ratio": "{:.3%}"},
    na_rep="",
)

Where in the loan's life the expected loss falls: the reasonable and supportable period, the reversion period, and the remaining life at long-run conditions.

In [ ]:
periods = {
    "reasonable_and_supportable": slice(0, RS_QUARTERS),
    "reversion": slice(RS_QUARTERS, RS_QUARTERS + REVERSION_QUARTERS),
    "remaining_life": slice(RS_QUARTERS + REVERSION_QUARTERS, None),
}
by_period = pd.DataFrame(
    {s: {p: r["loss"][:, sl].sum() for p, sl in periods.items()} for s, r in ecl_results.items()}
)

fig = go.Figure()
for s, r in ecl_results.items():
    loss = r["loss"].sum(axis=0)
    fig.add_trace(go.Scatter(x=np.arange(1, len(loss) + 1), y=loss / 1e6, name=s, line_color=COLORS[s]))
fig.add_vrect(x0=1, x1=RS_QUARTERS, fillcolor="grey", opacity=0.08, line_width=0)
fig.add_vrect(
    x0=RS_QUARTERS, x1=RS_QUARTERS + REVERSION_QUARTERS, fillcolor="grey", opacity=0.04, line_width=0
)
fig.update_layout(
    title="Expected loss by forecast quarter", xaxis_title="forecast quarter",
    yaxis_title="expected loss ($m)", height=350,
)
fig.show()

by_period.style.format("${:,.0f}")

<a id='toc6_3__'></a>

### Allowance by segment

In [ ]:
portfolio_segments = segments(portfolio)


def allowance_by(name):
    g = portfolio.groupby(portfolio_segments[name])
    return pd.DataFrame(
        {
            "loans": g.size(),
            "amortized_cost": g["balance"].sum(),
            "ecl": g["ecl"].sum(),
            "allowance_ratio": g["ecl"].sum() / g["balance"].sum(),
        }
    ).style.format({"amortized_cost": "${:,.0f}", "ecl": "${:,.0f}", "allowance_ratio": "{:.3%}"})


allowance_by("vintage")

In [ ]:
allowance_by("fico_band")

In [ ]:
allowance_by("current_ltv_band")

<a id='toc6_4__'></a>

### Champion and challenger allowance

The challenger runs through the same engine, portfolio and scenarios. The gap between the two allowances is model risk: the part of the estimate that depends on the choice of model rather than on the data or the economy.

In [ ]:
challenger_models = tuple(challenger[m] for m in MODEL_ORDER)
challenger_allowance, challenger_results = weighted_allowance(portfolio, scenario_paths, challenger_models)

pd.DataFrame(
    {
        set_name: {s: r[s]["ecl"].sum() / amortized_cost for s in SCENARIO_WEIGHTS}
        | {"probability_weighted": total / amortized_cost}
        for set_name, total, r in [
            ("champion", allowance, ecl_results),
            ("challenger", challenger_allowance, challenger_results),
        ]
    }
).style.format("{:.3%}")

<a id='toc7__'></a>

## Initialize the ValidMind inputs

Register the samples and models as ValidMind inputs so tests can run against them:

| Input | What it is |
|---|---|
| `train_panel`, `test_panel`, `oot_panel` | Loan-quarter samples with every default and prepayment model's predictions assigned |
| `train_defaults`, `test_defaults`, `oot_defaults` | Defaulted loan-quarters with both LGD models' predictions assigned |
| `reporting_portfolio` | The reporting-date portfolio with the champion's lifetime PD and ECL per scenario |
| `default_model`, `prepay_model`, `lgd_model` | The champion models |
| `challenger_default_model`, `challenger_prepay_model`, `challenger_lgd_model` | The challenger models |

In [ ]:
def to_vm_frame(df):
    # ValidMind inputs carry quarters as strings rather than pandas Periods.
    out = df.reset_index(drop=True).copy()
    for col in ["quarter", "orig_quarter"]:
        if col in out:
            out[col] = out[col].astype(str)
    return out


vm_models = {
    (set_name, m): vm.init_model(
        models[m], input_id=f"{m}_model" if set_name == "champion" else f"challenger_{m}_model"
    )
    for set_name, models in model_sets.items()
    for m in MODEL_ORDER
}

vm_panels = {
    name: vm.init_dataset(to_vm_frame(df), input_id=f"{name}_panel", target_column="defaulted")
    for name, df in [("train", train_panel), ("test", test_panel), ("oot", oot_panel)]
}
vm_defaults = {
    name: vm.init_dataset(to_vm_frame(df), input_id=f"{name}_defaults", target_column="lgd")
    for name, df in [("train", train_defaults), ("test", test_defaults), ("oot", oot_defaults)]
}
vm_portfolio = vm.init_dataset(
    to_vm_frame(portfolio),
    input_id="reporting_portfolio",
)

Assign each model's predictions to the samples it is tested on. Predictions are computed here with the models' own feature preparation, so tests read them from the dataset rather than re-scoring.

In [ ]:
for name, df in [("train", train_panel), ("test", test_panel), ("oot", oot_panel)]:
    X = model_features(df)
    for set_name, models in model_sets.items():
        for m in ["default", "prepay"]:
            probabilities = models[m].predict_proba(X)[:, 1]
            vm_panels[name].assign_predictions(
                model=vm_models[set_name, m],
                prediction_values=(probabilities > 0.5).astype(int),
                prediction_probabilities=probabilities,
            )

for name, df in [("train", train_defaults), ("test", test_defaults), ("oot", oot_defaults)]:
    for set_name, models in model_sets.items():
        vm_defaults[name].assign_predictions(
            model=vm_models[set_name, "lgd"], prediction_values=predict_lgd(models["lgd"], df)
        )

<a id='toc8__'></a>

## Run the CECL tests

The tests in `tests/`, next to this notebook, are written for CECL. Each one is a ValidMind test: a function that takes the inputs registered above, returns tables, figures and — where there is a clear bar — a pass/fail, and logs its result to the model's documentation.

| Test | What it checks |
|---|---|
| `QuarterlyCalibration` | Predicted vs actual event rate in every quarter, against a binomial band |
| `VintageBacktest` | Actual vs expected events by origination year |
| `DriverSensitivity` | The response to each driver, its sign, and whether it keeps moving beyond the development range |
| `ExtrapolationExposure` | How much of the projected portfolio falls outside the development range, per scenario |
| `LGDByLTVBand` | Realized vs predicted severity by mark-to-market LTV band |
| `WOEInformationValue` | Each driver's Weight of Evidence by bin and Information Value against default or prepayment |
| `ScenarioOrdering` | Loss rising with scenario severity, overall and per segment |
| `LifetimeLossBacktest` | The whole engine re-run at a past date on the realized economy, against realized losses |
| `ForecastPeriodSensitivity` | The allowance across forecast and reversion periods |
| `ScenarioWeightSensitivity` | The allowance across scenario weightings |
| `AllowanceAttribution` | Where in the loans' lives, and in which segments, the allowance sits |
| `MacroShockSensitivity` | The allowance under shocks to each macro driver, one at a time and combined, and whether it moves the right way |
| `HistoricalReplay` | Past downturns replayed on today's portfolio, against the severely adverse scenario |
| `ReverseStress` | How severe a combined unemployment and house-price shock must be to reach a multiple of the baseline allowance |

Register the folder as a test provider under the `cecl` namespace:

In [ ]:
from validmind.tests import LocalTestProvider, register_test_provider

cecl_tests = LocalTestProvider("tests")
register_test_provider("cecl", cecl_tests)
cecl_tests.list_tests()

<a id='toc8_1__'></a>

### Default and prepayment models

Weight of Evidence and Information Value on the train sample: how strongly, and in what shape, each driver separates defaults — and prepayments — from loans that stayed current. The test computes the bins and WoE directly with pandas.

In [ ]:
for target in ["defaulted", "prepaid"]:
    vm.tests.run_test(
        f"cecl.WOEInformationValue:{target}",
        inputs={"dataset": vm_panels["train"]},
        params={"target_column": target},
    ).log()

Quarterly calibration on the out-of-time sample, for default and for prepayment. `target_column` points the same test at either event.

In [ ]:
for m, target in [("default", "defaulted"), ("prepay", "prepaid")]:
    vm.tests.run_test(
        "cecl.QuarterlyCalibration",
        inputs={"dataset": vm_panels["oot"], "model": vm_models["champion", m]},
        params={"target_column": target},
    ).log()

Vintage back-test on the test sample: loans the model has not seen, from the same period. On the train sample the model would absorb vintage effects it was fitted to.

In [ ]:
vm.tests.run_test(
    "cecl.VintageBacktest",
    inputs={"dataset": vm_panels["test"], "model": vm_models["champion", "default"]},
).log()

The expected direction for each driver is the model's monotonic constraints. The test checks they hold, and shows what happens beyond the development range.

In [ ]:
for m, expected in [("default", DEFAULT_DIRECTION), ("prepay", PREPAY_DIRECTION)]:
    vm.tests.run_test(
        "cecl.DriverSensitivity",
        inputs={"dataset": vm_panels["train"], "model": vm_models["champion", m]},
        params={"expected_direction": expected},
    ).log()

In [ ]:
vm.tests.run_test(
    "cecl.ExtrapolationExposure",
    inputs={"datasets": [vm_portfolio, vm_panels["train"]]},
).log()

<a id='toc8_2__'></a>

### LGD model

In [ ]:
vm.tests.run_test(
    "cecl.LGDByLTVBand",
    inputs={"datasets": list(vm_defaults.values()), "model": vm_models["champion", "lgd"]},
).log()

<a id='toc8_3__'></a>

### Lifetime loss engine

The back-test books the allowance as of 2019Q4 — the end of the in-time window — and projects it on the economy that actually followed, so the pandemic, the refinance boom and the rate shock are all out of time.

In [ ]:
champion_vm_models = [vm_models["champion", m] for m in MODEL_ORDER]

vm.tests.run_test(
    "cecl.ScenarioOrdering",
    inputs={"dataset": vm_portfolio},
).log()

In [ ]:
vm.tests.run_test(
    "cecl.LifetimeLossBacktest",
    inputs={"datasets": list(vm_panels.values()), "models": champion_vm_models},
    params={"as_of": "2019Q4"},
).log()

<a id='toc8_4__'></a>

### Allowance assumptions

The forecast and reversion periods and the scenario weights are management judgments. These tests size their effect on the allowance. The forecast-period grid re-runs the lifetime engine nine times, so it takes a minute or two.

In [ ]:
vm.tests.run_test(
    "cecl.ForecastPeriodSensitivity",
    inputs={"dataset": vm_portfolio, "models": champion_vm_models},
).log()

In [ ]:
vm.tests.run_test(
    "cecl.ScenarioWeightSensitivity",
    inputs={"dataset": vm_portfolio},
).log()

In [ ]:
vm.tests.run_test(
    "cecl.AllowanceAttribution",
    inputs={"dataset": vm_portfolio, "models": champion_vm_models},
).log()

<a id='toc8_5__'></a>

### Stress testing

Three views of how the allowance behaves in a downturn, each re-running the lifetime engine on a stressed macro path built the same way as the Fed scenarios — a change from the reporting-quarter actuals over the reasonable and supportable period, then reversion:

- **Macro shock sensitivity** shocks one driver at a time from baseline, then unemployment and house prices together.
- **Historical replay** applies the moves of the early-1990s recession, the global financial crisis and the pandemic to today's portfolio and compares them with severely adverse.
- **Reverse stress** finds the combined unemployment and house-price shock that triples the baseline allowance.

Each runs the engine 6 to 15 times, about half a minute per test.

In [ ]:
vm.tests.run_test(
    "cecl.MacroShockSensitivity",
    inputs={"dataset": vm_portfolio, "models": champion_vm_models},
).log()

In [ ]:
vm.tests.run_test(
    "cecl.HistoricalReplay",
    inputs={"dataset": vm_portfolio, "models": champion_vm_models},
).log()

In [ ]:
vm.tests.run_test(
    "cecl.ReverseStress",
    inputs={"dataset": vm_portfolio, "models": champion_vm_models},
    params={"threshold_multiple": 3.0},
).log()

<a id='toc8_6__'></a>

### Challenger model

The same tests on the challenger, logged under a `challenger` result id. Calibration and the lifetime back-test show how the two model sets behave out of time; driver sensitivity checks whether the unconstrained logit found the expected signs on its own; the macro shocks and historical replay show how its linear extrapolation changes the stress results.

In [ ]:
challenger_vm_models = [vm_models["challenger", m] for m in MODEL_ORDER]

for m, target in [("default", "defaulted"), ("prepay", "prepaid")]:
    vm.tests.run_test(
        f"cecl.QuarterlyCalibration:challenger_{m}",
        inputs={"dataset": vm_panels["oot"], "model": vm_models["challenger", m]},
        params={"target_column": target},
    ).log()

for m, expected in [("default", DEFAULT_DIRECTION), ("prepay", PREPAY_DIRECTION)]:
    vm.tests.run_test(
        f"cecl.DriverSensitivity:challenger_{m}",
        inputs={"dataset": vm_panels["train"], "model": vm_models["challenger", m]},
        params={"expected_direction": expected},
    ).log()

vm.tests.run_test(
    "cecl.LifetimeLossBacktest:challenger",
    inputs={"datasets": list(vm_panels.values()), "models": challenger_vm_models},
    params={"as_of": "2019Q4"},
).log()

for test in ["MacroShockSensitivity", "HistoricalReplay"]:
    vm.tests.run_test(
        f"cecl.{test}:challenger",
        inputs={"dataset": vm_portfolio, "models": challenger_vm_models},
    ).log()

<!-- VALIDMIND COPYRIGHT -->

<small>

***

Copyright © 2023-2026 ValidMind Inc. All rights reserved.<br>
Refer to [LICENSE](https://github.com/validmind/validmind-library/blob/main/LICENSE) for details.<br>
SPDX-License-Identifier: AGPL-3.0 AND ValidMind Commercial</small>